# Human-in-the-loop (HITL)

**Module:** Module 3 — Production-Oriented Agent Design  
**Notebook:** `notebooks/module-3/3.3_hitl.ipynb`

## Purpose
Pause tool execution for a person to inspect an action, then approve, reject or edit the proposed arguments before resuming.

## Workflow
1. **Prepare the environment** — load required packages and local environment variables. Keep API keys in an ignored `.env` file; never paste credentials into notebook cells.
2. **Build the capability** — configure the model/agent, messages, tools, middleware or data source used in this lesson.
3. **Execute the example** — invoke or stream the agent and inspect the returned messages, tool calls, structured response or state.
4. **Validate behavior** — compare the result with the intended behavior, test an edge case, and check provider/service availability if execution fails.
5. **Reflect** — use the key learnings below to explain the design choices and limitations.

## Learning objectives
- Understand the concept demonstrated by this notebook and where it fits in an agent workflow.
- Explain the role of the main LangChain/LangGraph abstractions used in the implementation.
- Trace inputs through execution and interpret the resulting output.
- Identify operational concerns such as credentials, external services, state, safety and error handling.

## Key learnings
- Long conversations need deliberate context management, not simply unlimited message history.
- Human-in-the-loop checkpoints are a control boundary for actions with external side effects.
- Dynamic model/prompt/tool selection can reduce cost and tailor capabilities, but adds complexity.

## Prerequisites and execution notes
- Use the repository setup instructions in the root `README.md` and the supported Python version.
- Configure only the provider credentials needed by this notebook in your local `.env` file.
- Some cells call external APIs, local MCP services, audio devices or sample databases; those dependencies must be available.
- Model output is non-deterministic and provider model names/API behavior can change. Treat outputs as examples, not fixed expected strings.

---



### Implementation step 1

Imports the libraries and APIs needed by the following example: from dotenv import load_dotenv. Loads local environment variables so provider/API credentials can be read without hard-coding secrets in the notebook.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from dotenv import load_dotenv. Loads local environment variables so provider/API credentials can be read without hard-coding secrets in the notebook.

from dotenv import load_dotenv

load_dotenv()

### Implementation step 2

Imports the libraries and APIs needed by the following example: from langchain.tools import tool, ToolRuntime. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Uses runtime context to access request-scoped dependencies or contextual values without treating them as model-authored input.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.tools import tool, ToolRuntime. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Uses runtime context to access request-scoped dependencies or contextual values without treating them as model-authored input.

from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

### Implementation step 3

Imports the libraries and APIs needed by the following example: from langchain.agents import create_agent, AgentState; from langgraph.checkpoint.memory import InMemorySaver; from langchain.agents.middleware import HumanInTheLoopMiddleware. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint. Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.agents import create_agent, AgentState; from langgraph.checkpoint.memory import InMemorySaver; from langchain.agents.middleware import HumanInTheLoopMiddleware. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint. Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations.

from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="gpt-5-nano",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

### Implementation step 4

Imports the libraries and APIs needed by the following example: from langchain.messages import HumanMessage. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.messages import HumanMessage. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Enables checkpoint-backed short-term memory. Reuse the same thread identifier to continue a conversation; a new ID starts a separate thread.

from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

### Implementation step 5

Imports the libraries and APIs needed by the following example: from pprint import pprint. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from pprint import pprint. Inspects or prints the result so you can verify the output structure and follow the execution path.

from pprint import pprint

pprint(response)

### Implementation step 6

Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Inspects or prints the result so you can verify the output structure and follow the execution path.

print(response['__interrupt__'])

### Implementation step 7

Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Inspects or prints the result so you can verify the output structure and follow the execution path.

# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

## Approve

### Implementation step 9

Imports the libraries and APIs needed by the following example: from langgraph.types import Command. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langgraph.types import Command. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Inspects or prints the result so you can verify the output structure and follow the execution path.

from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

## Reject

### Implementation step 11

Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Inspects or prints the result so you can verify the output structure and follow the execution path.

response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

### Implementation step 12

Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Inspects or prints the result so you can verify the output structure and follow the execution path.

print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

## Edit

### Implementation step 14

Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Inspects or prints the result so you can verify the output structure and follow the execution path.

response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)